In [10]:
import numpy as np
import pandas as pd
import warnings

from itertools import product
from tqdm.auto import tqdm
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef
)

warnings.filterwarnings("ignore")

In [11]:
df = pd.read_csv(r"../../data/processed/data_vif.csv")
target = "Risk_Label"

# =========================
# 기본 정리
# =========================
# Date 컬럼 제거: 있으면 제거, 없으면 그냥 넘어감
if "Date" in df.columns:
    df = df.drop(columns=["Date"])

# Risk_Label 매핑: 문자형이면 0/1로 변환, 이미 0/1이면 그대로 사용
label_raw = df[target].copy()

if pd.api.types.is_numeric_dtype(label_raw):
    df[target] = label_raw.astype(int)
else:
    label_clean = label_raw.astype(str).str.strip().str.lower()

    df[target] = label_clean.map({
        "low risk": 0,
        "lowrisk": 0,
        "low": 0,
        "0": 0,
        "high risk": 1,
        "highrisk": 1,
        "high": 1,
        "1": 1
    })

    # 매핑 실패 여부 확인
    if df[target].isna().any():
        bad_values = label_raw[df[target].isna()].unique()
        raise ValueError(f"{target} 매핑 실패 값이 있습니다: {bad_values}")

    df[target] = df[target].astype(int)

# =========================
# 기존 train/valid/test = 45:35:20 유지
# 단, valid를 valid_fs / valid_tune으로 0.65:0.35 분할
# =========================
n_total = len(df)
n_train = int(n_total * 0.45)
n_valid = int(n_total * 0.35)
# 나머지는 test

train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df  = df.iloc[n_train + n_valid :].reset_index(drop=True)

# valid를 시간순으로 0.65:0.35 분할
valid_split = int(len(valid_df) * 0.65)
valid_fs_df   = valid_df.iloc[:valid_split].reset_index(drop=True)   # RF stepwise 변수선택 평가용
valid_tune_df = valid_df.iloc[valid_split:].reset_index(drop=True)   # hard voting 이후 최종 튜닝용

print(f"total rows: {n_total}")
print(f"train      : {len(train_df)}")
print(f"valid_fs   : {len(valid_fs_df)}  <- RF stepwise 변수선택용")
print(f"valid_tune : {len(valid_tune_df)}  <- 최종 모델 튜닝/cutoff 선택용")
print(f"test       : {len(test_df)}  <- 최종 평가용, 지금 단계에서는 사용 금지")

# =========================
# Split features/target
# =========================
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid_fs = valid_fs_df.drop(columns=[target])
y_valid_fs = valid_fs_df[target]

X_valid_tune = valid_tune_df.drop(columns=[target])
y_valid_tune = valid_tune_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

# 이후 최종 모델 튜닝 코드와 호환되게 alias를 만들어둠
# 단, RF stepwise에서는 X_valid/y_valid를 쓰지 말고 X_valid_fs/y_valid_fs를 쓸 것
X_valid = X_valid_tune
y_valid = y_valid_tune

print("\nshape")
print("X_train     :", X_train.shape, y_train.shape)
print("X_valid_fs  :", X_valid_fs.shape, y_valid_fs.shape)
print("X_valid_tune:", X_valid_tune.shape, y_valid_tune.shape)
print("X_test      :", X_test.shape, y_test.shape)

print("\nclass count")
print("train      :", y_train.value_counts().to_dict())
print("valid_fs   :", y_valid_fs.value_counts().to_dict())
print("valid_tune :", y_valid_tune.value_counts().to_dict())
# test label count는 최종 평가 전까지 확인하지 않음


total rows: 4108
train      : 1848
valid_fs   : 934  <- RF stepwise 변수선택용
valid_tune : 503  <- 최종 모델 튜닝/cutoff 선택용
test       : 823  <- 최종 평가용, 지금 단계에서는 사용 금지

shape
X_train     : (1848, 28) (1848,)
X_valid_fs  : (934, 28) (934,)
X_valid_tune: (503, 28) (503,)
X_test      : (823, 28) (823,)

class count
train      : {0: 1665, 1: 183}
valid_fs   : {0: 842, 1: 92}
valid_tune : {0: 433, 1: 70}


In [12]:
scaler = MinMaxScaler().set_output(transform="pandas")

# train에만 fit하고 valid_fs / valid_tune / test에는 같은 scaler 적용
X_train = scaler.fit_transform(X_train)
X_valid_fs = scaler.transform(X_valid_fs)
X_valid_tune = scaler.transform(X_valid_tune)
X_test = scaler.transform(X_test)

# 이후 최종 모델 튜닝 코드와 호환되게 alias 유지
X_valid = X_valid_tune

print("scaled shape")
print("X_train     :", X_train.shape)
print("X_valid_fs  :", X_valid_fs.shape)
print("X_valid_tune:", X_valid_tune.shape)
print("X_test      :", X_test.shape)

print("\nclass count")
print("y_train     :", y_train.value_counts().to_dict())
print("y_valid_fs  :", y_valid_fs.value_counts().to_dict())
print("y_valid_tune:", y_valid_tune.value_counts().to_dict())
# test label count는 최종 평가 전까지 확인하지 않음


scaled shape
X_train     : (1848, 28)
X_valid_fs  : (934, 28)
X_valid_tune: (503, 28)
X_test      : (823, 28)

class count
y_train     : {0: 1665, 1: 183}
y_valid_fs  : {0: 842, 1: 92}
y_valid_tune: {0: 433, 1: 70}


In [13]:
# =========================
# RF Stepwise Selection Fixed Parameters
# =========================
# MultiSURF에서 사용하는 RF와 동일하게 고정
# 이 RF는 "최종 예측모델"이 아니라 "변수선택용 평가 모델"임.

RF_FIXED_PARAMS = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_leaf": 5,
    "min_samples_split": 2,
    "max_features": "sqrt",
    "class_weight": "balanced"
}

# 기존 stepwise_selection_rf 함수가 rf_param_list를 받는 구조라서
# 고정 파라미터 1개를 리스트로 감싸서 넣음
rf_param_list = [RF_FIXED_PARAMS]

# cutoff 후보
threshold_grid = np.round(np.arange(0.05, 0.81, 0.05), 2).tolist()

print(f"RF parameter combinations: {len(rf_param_list)}")
print("RF_FIXED_PARAMS:", RF_FIXED_PARAMS)
print(f"Threshold candidates: {len(threshold_grid)}")
print(f"Total metric evaluations per feature subset: {len(rf_param_list) * len(threshold_grid)}")

RF parameter combinations: 1
RF_FIXED_PARAMS: {'n_estimators': 500, 'max_depth': None, 'min_samples_leaf': 5, 'min_samples_split': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'}
Threshold candidates: 16
Total metric evaluations per feature subset: 16


## Random Forest 기반 Wrapper

각 단계에서 후보 변수 subset을 Random Forest로 학습하고, validation G-Mean을 기준으로 추가/제거 여부를 결정한다.


In [14]:
# =========================
# Metric Functions
# =========================

def get_metrics(y_true, y_prob, threshold):
    """
    y_prob에 threshold를 적용하여 이진분류 성능지표 계산.
    Positive class = 1, 즉 High Risk.
    """
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = np.sqrt(recall * spec)

    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        pr_auc = np.nan

    return {
        "threshold": threshold,
        "acc": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "spec": spec,
        "gmean": gmean,
        "balanced_acc": balanced_acc,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def metric_key(metrics):
    """
    변수 subset 비교 기준.
    1순위: G-Mean
    2순위: Recall
    3순위: F1
    4순위: Accuracy
    5순위: Precision
    """
    return (
        metrics.get("gmean", -np.inf),
        metrics.get("recall", -np.inf),
        metrics.get("f1", -np.inf),
        metrics.get("acc", -np.inf),
        metrics.get("precision", -np.inf)
    )


# =========================
# Feature Unit Functions
# =========================

def build_feature_units(
    feature_names,
    group_prefixes=("Signal1", "Signal2", "Signal3", "Signal4")
):
    """
    Signal 더미변수는 그룹 단위로 묶고,
    나머지 변수는 개별 변수 단위로 둠.

    예:
    Signal4_Buy + Signal4_Sell -> Signal4라는 하나의 선택 단위

    _Stay 컬럼은 기준범주로 보고 선택 단위에서 제외함.
    """
    feature_names = list(feature_names)

    feature_units = {}
    grouped_cols = set()
    ignored_cols = set()

    for prefix in group_prefixes:
        cols = [
            c for c in feature_names
            if ((c == prefix) or c.startswith(prefix + "_"))
            and not c.endswith("_Stay")
        ]

        stay_cols = [
            c for c in feature_names
            if ((c == prefix) or c.startswith(prefix + "_"))
            and c.endswith("_Stay")
        ]

        ignored_cols.update(stay_cols)

        if len(cols) > 0:
            feature_units[prefix] = cols
            grouped_cols.update(cols)

    for col in feature_names:
        if col in grouped_cols:
            continue
        if col in ignored_cols:
            continue
        if col.endswith("_Stay"):
            continue

        feature_units[col] = [col]

    return feature_units


def flatten_units(units, feature_units):
    """
    선택 단위 목록을 실제 컬럼 목록으로 변환.
    """
    cols = []

    for unit in units:
        cols.extend(feature_units[unit])

    # 중복 제거, 순서 유지
    seen = set()
    unique_cols = []

    for c in cols:
        if c not in seen:
            unique_cols.append(c)
            seen.add(c)

    return unique_cols


# =========================
# RF Evaluation Function
# =========================

def fit_eval_rf(
    X_train,
    y_train,
    X_valid,
    y_valid,
    features,
    rf_param_list,
    threshold_grid,
    random_state=1,
    n_jobs=-1,
    cache=None
):
    """
    특정 feature subset에 대해 RF를 학습하고 valid 성능 평가.

    중요:
    - RF는 파라미터 조합별로 1번만 학습
    - threshold마다 재학습하지 않음
    - 학습된 y_prob에 threshold만 바꿔가며 평가
    """
    features = list(features)

    if len(features) == 0:
        raise ValueError("features가 비어 있습니다.")

    if cache is None:
        cache = {}

    best_record = None

    for rf_params in rf_param_list:
        param_key = tuple(sorted(rf_params.items()))
        feature_key = tuple(sorted(features))
        cache_key = (feature_key, param_key)

        if cache_key in cache:
            y_prob = cache[cache_key]
        else:
            model = RandomForestClassifier(
                **rf_params,
                random_state=random_state,
                n_jobs=n_jobs
            )

            model.fit(X_train[features], y_train)
            y_prob = model.predict_proba(X_valid[features])[:, 1]
            cache[cache_key] = y_prob

        for threshold in threshold_grid:
            metrics = get_metrics(y_valid, y_prob, threshold)

            record = {
                "features": features,
                "params": {
                    **rf_params,
                    "threshold": threshold
                },
                "metrics": metrics
            }

            if best_record is None or metric_key(metrics) > metric_key(best_record["metrics"]):
                best_record = record

    return best_record


def _empty_result(method_name):
    return {
        "method": method_name,
        "selected_units": [],
        "selected_features": [],
        "best_params": {
            "threshold": np.nan
        },
        "best_metrics": {
            "gmean": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "acc": 0.0,
            "precision": 0.0,
            "spec": 0.0,
            "balanced_acc": 0.0,
            "mcc": 0.0,
            "roc_auc": np.nan,
            "pr_auc": np.nan,
            "tn": 0,
            "fp": 0,
            "fn": 0,
            "tp": 0
        },
        "step_history": pd.DataFrame(),
        "candidate_history": pd.DataFrame()
    }


def _update_best_seen(
    candidate_units,
    candidate_features,
    candidate_params,
    candidate_metrics,
    best_seen
):
    """
    경로 중 최고 성능 변수셋 갱신.
    best_seen이 None이거나 candidate가 더 좋으면 갱신.
    """
    if best_seen is None:
        return {
            "units": candidate_units.copy(),
            "features": candidate_features.copy(),
            "params": candidate_params.copy(),
            "metrics": candidate_metrics.copy()
        }, True

    if metric_key(candidate_metrics) > metric_key(best_seen["metrics"]):
        return {
            "units": candidate_units.copy(),
            "features": candidate_features.copy(),
            "params": candidate_params.copy(),
            "metrics": candidate_metrics.copy()
        }, True

    return best_seen, False


# =========================
# Forward Selection
# =========================

def forward_selection_rf(
    X_train,
    y_train,
    X_valid,
    y_valid,
    feature_units,
    rf_param_list,
    threshold_grid,
    min_delta=None,
    max_units_to_select=None,
    random_state=1,
    n_jobs=-1,
    verbose=True
):
    """
    RF 기반 전진선택 - 전체 경로 탐색 후 최대 G-Mean 선택 버전.

    기존 방식:
    - G-Mean이 개선되지 않으면 중단

    현재 방식:
    - 남은 후보가 없어질 때까지 계속 하나씩 추가
    - 각 step에서 가장 좋은 추가 후보를 선택
    - 전체 추가 경로 중 valid1 G-Mean이 가장 높은 변수셋을 최종 Forward 결과로 반환

    min_delta는 호환성을 위해 인자로만 남겨두며, 중단 기준으로 사용하지 않음.
    """
    selected_units = []
    remaining_units = list(feature_units.keys())

    step_history = []
    candidate_history = []
    eval_cache = {}

    best_seen = None
    current_score = -np.inf

    step = 1

    while len(remaining_units) > 0:
        if max_units_to_select is not None and len(selected_units) >= max_units_to_select:
            if verbose:
                print(f"\n[FORWARD] max_units_to_select={max_units_to_select} 도달, 종료")
            break

        if verbose:
            print(f"\n[FORWARD STEP {step}]")
            print(f"현재 선택 단위 수: {len(selected_units)} | 추가 후보 단위 수: {len(remaining_units)}")

        forward_best = None

        for add_unit in tqdm(remaining_units, disable=not verbose, desc=f"FORWARD {step} add"):
            trial_units = selected_units + [add_unit]
            trial_features = flatten_units(trial_units, feature_units)

            record = fit_eval_rf(
                X_train, y_train, X_valid, y_valid,
                features=trial_features,
                rf_param_list=rf_param_list,
                threshold_grid=threshold_grid,
                random_state=random_state,
                n_jobs=n_jobs,
                cache=eval_cache
            )

            candidate_history.append({
                "method": "forward",
                "step": step,
                "action": "add_candidate",
                "unit": add_unit,
                "columns": " | ".join(feature_units[add_unit]),
                "n_units": len(trial_units),
                "n_features": len(trial_features),
                **record["params"],
                **record["metrics"]
            })

            candidate = {
                "unit": add_unit,
                "units": trial_units,
                **record
            }

            if forward_best is None or metric_key(record["metrics"]) > metric_key(forward_best["metrics"]):
                forward_best = candidate

        # 이번 step에서 가장 좋은 후보를 무조건 추가
        added_unit = forward_best["unit"]
        previous_score = current_score

        selected_units = forward_best["units"]
        remaining_units.remove(added_unit)

        current_features = forward_best["features"]
        current_params = forward_best["params"]
        current_metrics = forward_best["metrics"]
        current_score = current_metrics["gmean"]

        improvement = (
            current_score - previous_score
            if np.isfinite(previous_score)
            else np.nan
        )

        best_seen, is_best_seen = _update_best_seen(
            selected_units,
            current_features,
            current_params,
            current_metrics,
            best_seen
        )

        step_history.append({
            "method": "forward",
            "step": step,
            "action": "add",
            "unit": added_unit,
            "columns": " | ".join(feature_units[added_unit]),
            "n_units": len(selected_units),
            "n_features": len(current_features),
            "improvement": improvement,
            "is_best_seen": is_best_seen,
            **current_params,
            **current_metrics
        })

        if verbose:
            marker = "  <-- 현재까지 최고" if is_best_seen else ""
            print(
                f"추가 단위: {added_unit} | "
                f"gmean={current_score:.4f} | "
                f"improvement={improvement if np.isfinite(improvement) else np.nan:.6f} | "
                f"recall={current_metrics['recall']:.4f} | "
                f"f1={current_metrics['f1']:.4f} | "
                f"threshold={current_params['threshold']}"
                f"{marker}"
            )

        step += 1

    if best_seen is None:
        return _empty_result("forward")

    if verbose:
        print("\n" + "=" * 80)
        print("[FORWARD 최종 선택]")
        print("전체 추가 경로 중 valid1 G-Mean이 최대인 지점 선택")
        print("=" * 80)
        print("best_seen_units:", len(best_seen["units"]))
        print("best_seen_features:", len(best_seen["features"]))
        print("best_seen_gmean:", best_seen["metrics"]["gmean"])
        print("best_seen_threshold:", best_seen["params"]["threshold"])

    return {
        "method": "forward",
        "selected_units": best_seen["units"],
        "selected_features": best_seen["features"],
        "best_params": best_seen["params"],
        "best_metrics": best_seen["metrics"],
        "step_history": pd.DataFrame(step_history),
        "candidate_history": pd.DataFrame(candidate_history)
    }


# =========================
# Backward Elimination
# =========================

def backward_elimination_rf(
    X_train,
    y_train,
    X_valid,
    y_valid,
    feature_units,
    rf_param_list,
    threshold_grid,
    backward_drop_tolerance=None,
    min_units_to_keep=1,
    random_state=1,
    n_jobs=-1,
    verbose=True
):
    """
    RF 기반 후진제거 - 전체 경로 탐색 후 최대 G-Mean 선택 버전.

    현재 방식:
    - 전체 변수셋에서 시작
    - 매 step에서 제거 후 성능이 가장 좋은 변수를 하나 제거
    - min_units_to_keep까지 계속 제거
    - 전체 제거 경로 중 valid1 G-Mean이 가장 높은 변수셋을 최종 Backward 결과로 반환

    backward_drop_tolerance는 호환성을 위해 인자로만 남겨두며, 중단 기준으로 사용하지 않음.
    """
    selected_units = list(feature_units.keys())

    step_history = []
    candidate_history = []
    eval_cache = {}

    best_seen = None

    # 전체 변수셋 성능도 경로의 첫 지점으로 평가
    current_features = flatten_units(selected_units, feature_units)

    current_record = fit_eval_rf(
        X_train, y_train, X_valid, y_valid,
        features=current_features,
        rf_param_list=rf_param_list,
        threshold_grid=threshold_grid,
        random_state=random_state,
        n_jobs=n_jobs,
        cache=eval_cache
    )

    current_params = current_record["params"]
    current_metrics = current_record["metrics"]
    current_score = current_metrics["gmean"]

    best_seen, is_best_seen = _update_best_seen(
        selected_units,
        current_features,
        current_params,
        current_metrics,
        best_seen
    )

    step_history.append({
        "method": "backward",
        "step": 0,
        "action": "start",
        "unit": "__FULL_SET__",
        "columns": " | ".join(current_features),
        "n_units": len(selected_units),
        "n_features": len(current_features),
        "improvement": np.nan,
        "is_best_seen": is_best_seen,
        **current_params,
        **current_metrics
    })

    if verbose:
        print("\n[BACKWARD INITIAL]")
        print(f"초기 선택 단위 수: {len(selected_units)}")
        print(
            f"initial gmean={current_score:.4f} | "
            f"recall={current_metrics['recall']:.4f} | "
            f"f1={current_metrics['f1']:.4f} | threshold={current_params['threshold']}"
        )

    step = 1

    while len(selected_units) > min_units_to_keep:
        if verbose:
            print(f"\n[BACKWARD STEP {step}]")
            print(f"현재 선택 단위 수: {len(selected_units)}")

        backward_best = None

        for remove_unit in tqdm(selected_units, disable=not verbose, desc=f"BACKWARD {step} remove"):
            trial_units = [u for u in selected_units if u != remove_unit]
            trial_features = flatten_units(trial_units, feature_units)

            record = fit_eval_rf(
                X_train, y_train, X_valid, y_valid,
                features=trial_features,
                rf_param_list=rf_param_list,
                threshold_grid=threshold_grid,
                random_state=random_state,
                n_jobs=n_jobs,
                cache=eval_cache
            )

            candidate_history.append({
                "method": "backward",
                "step": step,
                "action": "remove_candidate",
                "unit": remove_unit,
                "columns": " | ".join(feature_units[remove_unit]),
                "n_units": len(trial_units),
                "n_features": len(trial_features),
                **record["params"],
                **record["metrics"]
            })

            candidate = {
                "unit": remove_unit,
                "units": trial_units,
                **record
            }

            if backward_best is None or metric_key(record["metrics"]) > metric_key(backward_best["metrics"]):
                backward_best = candidate

        # 이번 step에서 제거 후 성능이 가장 좋은 후보를 무조건 제거
        removed_unit = backward_best["unit"]
        previous_score = current_score

        selected_units = backward_best["units"]

        current_features = backward_best["features"]
        current_params = backward_best["params"]
        current_metrics = backward_best["metrics"]
        current_score = current_metrics["gmean"]

        improvement = current_score - previous_score

        best_seen, is_best_seen = _update_best_seen(
            selected_units,
            current_features,
            current_params,
            current_metrics,
            best_seen
        )

        step_history.append({
            "method": "backward",
            "step": step,
            "action": "remove",
            "unit": removed_unit,
            "columns": " | ".join(feature_units[removed_unit]),
            "n_units": len(selected_units),
            "n_features": len(current_features),
            "improvement": improvement,
            "is_best_seen": is_best_seen,
            **current_params,
            **current_metrics
        })

        if verbose:
            marker = "  <-- 현재까지 최고" if is_best_seen else ""
            print(
                f"제거 단위: {removed_unit} | "
                f"gmean={current_score:.4f} | "
                f"improvement={improvement:.6f} | "
                f"threshold={current_params['threshold']}"
                f"{marker}"
            )

        step += 1

    if best_seen is None:
        return _empty_result("backward")

    if verbose:
        print("\n" + "=" * 80)
        print("[BACKWARD 최종 선택]")
        print("전체 제거 경로 중 valid1 G-Mean이 최대인 지점 선택")
        print("=" * 80)
        print("best_seen_units:", len(best_seen["units"]))
        print("best_seen_features:", len(best_seen["features"]))
        print("best_seen_gmean:", best_seen["metrics"]["gmean"])
        print("best_seen_threshold:", best_seen["params"]["threshold"])

    return {
        "method": "backward",
        "selected_units": best_seen["units"],
        "selected_features": best_seen["features"],
        "best_params": best_seen["params"],
        "best_metrics": best_seen["metrics"],
        "step_history": pd.DataFrame(step_history),
        "candidate_history": pd.DataFrame(candidate_history)
    }


# =========================
# Stepwise Selection
# =========================

def stepwise_selection_rf(
    X_train,
    y_train,
    X_valid,
    y_valid,
    feature_units,
    rf_param_list,
    threshold_grid,
    min_delta=None,
    backward_drop_tolerance=None,
    max_units_to_select=None,
    max_steps=None,
    random_state=1,
    n_jobs=-1,
    verbose=True
):
    """
    RF 기반 단계적선택 - 경로 탐색 후 최대 G-Mean 선택 버전.

    현재 방식:
    - Forward 방향으로 후보를 계속 추가
    - 각 추가 후, 제거하면 현재 성능이 개선되는 경우에만 후진 제거 수행
    - 모든 step에서 valid1 G-Mean 기록
    - 전체 stepwise 경로 중 valid1 G-Mean이 가장 높은 변수셋을 최종 Stepwise 결과로 반환

    min_delta, backward_drop_tolerance는 호환성을 위해 인자로만 남겨두며, 중단 기준으로 사용하지 않음.

    무한 반복 방지:
    - 이미 방문한 변수 조합은 다시 방문하지 않음.
    - max_steps로 강제 종료.
    """
    selected_units = []
    remaining_units = list(feature_units.keys())

    step_history = []
    candidate_history = []
    eval_cache = {}

    best_seen = None

    current_params = None
    current_metrics = None
    current_score = -np.inf

    seen_states = {tuple()}

    if max_steps is None:
        max_steps = max(10, 5 * len(feature_units))

    step = 1

    while len(remaining_units) > 0:
        if step > max_steps:
            if verbose:
                print(f"\n[STEPWISE] max_steps={max_steps} 도달, 종료")
            break

        if max_units_to_select is not None and len(selected_units) >= max_units_to_select:
            if verbose:
                print(f"\n[STEPWISE] max_units_to_select={max_units_to_select} 도달, 종료")
            break

        # =========================
        # 1) Forward add
        # =========================
        if verbose:
            print(f"\n[STEPWISE STEP {step} - FORWARD]")
            print(f"현재 선택 단위 수: {len(selected_units)} | 추가 후보 단위 수: {len(remaining_units)}")

        forward_best = None

        for add_unit in tqdm(remaining_units, disable=not verbose, desc=f"STEPWISE {step} add"):
            trial_units = selected_units + [add_unit]
            trial_state = tuple(sorted(trial_units))

            if trial_state in seen_states:
                continue

            trial_features = flatten_units(trial_units, feature_units)

            record = fit_eval_rf(
                X_train, y_train, X_valid, y_valid,
                features=trial_features,
                rf_param_list=rf_param_list,
                threshold_grid=threshold_grid,
                random_state=random_state,
                n_jobs=n_jobs,
                cache=eval_cache
            )

            candidate_history.append({
                "method": "stepwise",
                "step": step,
                "action": "add_candidate",
                "unit": add_unit,
                "columns": " | ".join(feature_units[add_unit]),
                "n_units": len(trial_units),
                "n_features": len(trial_features),
                **record["params"],
                **record["metrics"]
            })

            candidate = {
                "unit": add_unit,
                "units": trial_units,
                "state": trial_state,
                **record
            }

            if forward_best is None or metric_key(record["metrics"]) > metric_key(forward_best["metrics"]):
                forward_best = candidate

        if forward_best is None:
            if verbose:
                print("추가 가능한 미방문 조합이 없어서 종료")
            break

        added_unit = forward_best["unit"]
        previous_score = current_score

        selected_units = forward_best["units"]
        remaining_units.remove(added_unit)
        seen_states.add(tuple(sorted(selected_units)))

        current_features = forward_best["features"]
        current_params = forward_best["params"]
        current_metrics = forward_best["metrics"]
        current_score = current_metrics["gmean"]

        improvement = (
            current_score - previous_score
            if np.isfinite(previous_score)
            else np.nan
        )

        best_seen, is_best_seen = _update_best_seen(
            selected_units,
            current_features,
            current_params,
            current_metrics,
            best_seen
        )

        step_history.append({
            "method": "stepwise",
            "step": step,
            "action": "add",
            "unit": added_unit,
            "columns": " | ".join(feature_units[added_unit]),
            "n_units": len(selected_units),
            "n_features": len(current_features),
            "improvement": improvement,
            "is_best_seen": is_best_seen,
            **current_params,
            **current_metrics
        })

        if verbose:
            marker = "  <-- 현재까지 최고" if is_best_seen else ""
            print(
                f"추가 단위: {added_unit} | "
                f"gmean={current_score:.4f} | "
                f"improvement={improvement if np.isfinite(improvement) else np.nan:.6f} | "
                f"recall={current_metrics['recall']:.4f} | "
                f"f1={current_metrics['f1']:.4f} | "
                f"threshold={current_params['threshold']}"
                f"{marker}"
            )

        # =========================
        # 2) Backward prune
        # =========================
        # Stepwise에서 제거는 "현재 상태보다 성능이 개선될 때만" 수행.
        # 단, 전체 경로 중 최고 변수셋은 계속 기록함.
        backward_improved = True

        while backward_improved and len(selected_units) > 1:
            backward_improved = False
            backward_best = None

            if verbose:
                print(f"\n[STEPWISE STEP {step} - BACKWARD]")
                print(f"현재 선택 단위 수: {len(selected_units)}")

            for remove_unit in tqdm(selected_units, disable=not verbose, desc=f"STEPWISE {step} remove"):
                trial_units = [u for u in selected_units if u != remove_unit]
                trial_state = tuple(sorted(trial_units))

                if trial_state in seen_states:
                    continue

                trial_features = flatten_units(trial_units, feature_units)

                record = fit_eval_rf(
                    X_train, y_train, X_valid, y_valid,
                    features=trial_features,
                    rf_param_list=rf_param_list,
                    threshold_grid=threshold_grid,
                    random_state=random_state,
                    n_jobs=n_jobs,
                    cache=eval_cache
                )

                candidate_history.append({
                    "method": "stepwise",
                    "step": step,
                    "action": "remove_candidate",
                    "unit": remove_unit,
                    "columns": " | ".join(feature_units[remove_unit]),
                    "n_units": len(trial_units),
                    "n_features": len(trial_features),
                    **record["params"],
                    **record["metrics"]
                })

                candidate = {
                    "unit": remove_unit,
                    "units": trial_units,
                    "state": trial_state,
                    **record
                }

                if backward_best is None or metric_key(record["metrics"]) > metric_key(backward_best["metrics"]):
                    backward_best = candidate

            if backward_best is None:
                if verbose:
                    print("제거 가능한 미방문 조합이 없음")
                break

            # 제거는 현재 성능보다 좋아질 때만 허용
            if metric_key(backward_best["metrics"]) > metric_key(current_metrics):
                removed_unit = backward_best["unit"]
                previous_score = current_score

                selected_units = backward_best["units"]
                seen_states.add(tuple(sorted(selected_units)))

                if removed_unit not in remaining_units:
                    remaining_units.append(removed_unit)

                current_features = backward_best["features"]
                current_params = backward_best["params"]
                current_metrics = backward_best["metrics"]
                current_score = current_metrics["gmean"]

                improvement = current_score - previous_score

                best_seen, is_best_seen = _update_best_seen(
                    selected_units,
                    current_features,
                    current_params,
                    current_metrics,
                    best_seen
                )

                backward_improved = True

                step_history.append({
                    "method": "stepwise",
                    "step": step,
                    "action": "remove",
                    "unit": removed_unit,
                    "columns": " | ".join(feature_units[removed_unit]),
                    "n_units": len(selected_units),
                    "n_features": len(current_features),
                    "improvement": improvement,
                    "is_best_seen": is_best_seen,
                    **current_params,
                    **current_metrics
                })

                if verbose:
                    marker = "  <-- 현재까지 최고" if is_best_seen else ""
                    print(
                        f"제거 단위: {removed_unit} | "
                        f"gmean={current_score:.4f} | "
                        f"improvement={improvement:.6f} | "
                        f"threshold={current_params['threshold']}"
                        f"{marker}"
                    )
            else:
                if verbose:
                    print("제거해도 현재 stepwise 성능이 개선되지 않으므로 backward prune 종료")
                break

        step += 1

    if best_seen is None:
        return _empty_result("stepwise")

    if verbose:
        print("\n" + "=" * 80)
        print("[STEPWISE 최종 선택]")
        print("전체 stepwise 경로 중 valid1 G-Mean이 최대인 지점 선택")
        print("=" * 80)
        print("best_seen_units:", len(best_seen["units"]))
        print("best_seen_features:", len(best_seen["features"]))
        print("best_seen_gmean:", best_seen["metrics"]["gmean"])
        print("best_seen_threshold:", best_seen["params"]["threshold"])

    return {
        "method": "stepwise",
        "selected_units": best_seen["units"],
        "selected_features": best_seen["features"],
        "best_params": best_seen["params"],
        "best_metrics": best_seen["metrics"],
        "step_history": pd.DataFrame(step_history),
        "candidate_history": pd.DataFrame(candidate_history)
    }


# =========================
# valid2에서 Forward / Backward / Stepwise 비교
# =========================

def evaluate_rf_wrapper_methods_on_valid2(
    selection_results,
    X_train_eval,
    y_train_eval,
    X_valid2,
    y_valid2,
    rf_param_list,
    threshold_grid,
    random_state=1,
    n_jobs=-1
):
    """
    valid1에서 생성된 Forward / Backward / Stepwise 변수셋을
    valid2에서 같은 RF 평가 절차로 비교.
    """
    rows = []
    eval_cache = {}

    for method_name, result in selection_results.items():
        selected_features = result["selected_features"]

        if len(selected_features) == 0:
            rows.append({
                "method": method_name,
                "n_units": 0,
                "n_features": 0,
                "selected_units": "",
                "selected_features": "",
                "threshold": np.nan,
                "gmean": 0.0,
                "recall": 0.0,
                "f1": 0.0,
                "acc": 0.0,
                "precision": 0.0,
                "spec": 0.0,
                "balanced_acc": 0.0,
                "mcc": 0.0,
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "tn": 0,
                "fp": 0,
                "fn": 0,
                "tp": 0
            })
            continue

        record = fit_eval_rf(
            X_train_eval,
            y_train_eval,
            X_valid2,
            y_valid2,
            features=selected_features,
            rf_param_list=rf_param_list,
            threshold_grid=threshold_grid,
            random_state=random_state,
            n_jobs=n_jobs,
            cache=eval_cache
        )

        rows.append({
            "method": method_name,
            "n_units": len(result["selected_units"]),
            "n_features": len(selected_features),
            "selected_units": " | ".join(result["selected_units"]),
            "selected_features": " | ".join(selected_features),
            **record["params"],
            **record["metrics"]
        })

    comparison_df = pd.DataFrame(rows)

    comparison_df = comparison_df.sort_values(
        ["gmean", "recall", "f1", "acc", "precision"],
        ascending=False
    ).reset_index(drop=True)

    return comparison_df

### RF Wrapper 변수선택 공통 준비

In [15]:
# =========================
# RF Wrapper 변수선택 공통 준비
# =========================
# 흐름:
# 1. valid_fs(valid1)에서 Forward / Backward / Stepwise 각각 변수셋 생성
# 2. valid_tune(valid2)에서 세 변수셋을 비교
# 3. valid2 G-Mean이 가장 좋은 방식의 변수셋을 RF wrapper 최종 결과로 사용
# 4. test는 사용하지 않음

# Signal 더미변수는 그룹 단위로 묶고, 나머지는 개별 변수 단위로 둠
feature_units = build_feature_units(
    X_train.columns,
    group_prefixes=("Signal1", "Signal2", "Signal3", "Signal4")
)

print("=" * 80)
print("Feature selection units")
print("=" * 80)
print(f"전체 실제 컬럼 수: {X_train.shape[1]}")
print(f"선택 단위 수: {len(feature_units)}")

signal_units = {k: v for k, v in feature_units.items() if k.startswith("Signal")}
print("\nSignal 그룹 단위:")
if signal_units:
    for k, v in signal_units.items():
        print(f"{k}: {v}")
else:
    print("Signal 그룹 컬럼이 없습니다.")

# 결과 저장용 dictionary
selection_results = {}

Feature selection units
전체 실제 컬럼 수: 28
선택 단위 수: 24

Signal 그룹 단위:
Signal1: ['Signal1_Buy', 'Signal1_Sell']
Signal2: ['Signal2_Buy', 'Signal2_Sell']
Signal3: ['Signal3_Buy', 'Signal3_Sell']
Signal4: ['Signal4_Buy', 'Signal4_Sell']


### Forward Selection 실행

In [16]:
# =========================
# 1) RF Forward Selection
# =========================

forward_result = forward_selection_rf(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid_fs,
    y_valid=y_valid_fs,
    feature_units=feature_units,
    rf_param_list=rf_param_list,
    threshold_grid=threshold_grid,
    min_delta=0,
    max_units_to_select=None,
    random_state=1,
    n_jobs=-1,
    verbose=True
)

selection_results["forward"] = forward_result

print("\n" + "=" * 80)
print("Forward Selection 결과")
print("=" * 80)
print("선택 단위 수:", len(forward_result["selected_units"]))
print("실제 컬럼 수:", len(forward_result["selected_features"]))
print("best threshold:", forward_result["best_params"]["threshold"])
print("valid1 gmean:", forward_result["best_metrics"]["gmean"])
print("valid1 recall:", forward_result["best_metrics"]["recall"])
print("valid1 f1:", forward_result["best_metrics"]["f1"])

print("\nselected_units:")
print(forward_result["selected_units"])

print("\nselected_features:")
print(forward_result["selected_features"])


[FORWARD STEP 1]
현재 선택 단위 수: 0 | 추가 후보 단위 수: 24


FORWARD 1 add: 100%|██████████| 24/24 [00:17<00:00,  1.41it/s]


추가 단위: NASDAQ_return(%) | gmean=0.6239 | improvement=nan | recall=0.5761 | f1=0.2536 | threshold=0.3  <-- 현재까지 최고

[FORWARD STEP 2]
현재 선택 단위 수: 1 | 추가 후보 단위 수: 23


FORWARD 2 add: 100%|██████████| 23/23 [00:15<00:00,  1.47it/s]


추가 단위: KOSPI 200_PPO | gmean=0.6646 | improvement=0.040704 | recall=0.6304 | f1=0.2886 | threshold=0.25  <-- 현재까지 최고

[FORWARD STEP 3]
현재 선택 단위 수: 2 | 추가 후보 단위 수: 22


...

추가 단위: KOSPI 200 lagged_1_return(%) | gmean=0.6764 | improvement=-0.006364 | recall=0.6304 | f1=0.3045 | threshold=0.2

[FORWARD STEP 23]
현재 선택 단위 수: 22 | 추가 후보 단위 수: 2


FORWARD 23 add: 100%|██████████| 2/2 [00:01<00:00,  1.43it/s]


추가 단위: Gold Spot_return(%) | gmean=0.6861 | improvement=0.009702 | recall=0.6413 | f1=0.3147 | threshold=0.2

[FORWARD STEP 24]
현재 선택 단위 수: 23 | 추가 후보 단위 수: 1


FORWARD 24 add: 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]

추가 단위: KOSDAQ_return(%) | gmean=0.6727 | improvement=-0.013367 | recall=0.6196 | f1=0.3032 | threshold=0.2

[FORWARD 최종 선택]
전체 추가 경로 중 valid1 G-Mean이 최대인 지점 선택
best_seen_units: 11
best_seen_features: 13
best_seen_gmean: 0.7175274250512746
best_seen_threshold: 0.2

Forward Selection 결과
선택 단위 수: 11
실제 컬럼 수: 13
best threshold: 0.2
valid1 gmean: 0.7175274250512746
valid1 recall: 0.75
valid1 f1: 0.3247058823529412

selected_units:
['NASDAQ_return(%)', 'KOSPI 200_PPO', 'KOSPI 200 Volume', 'Brent Crude Oil_return(%)', 'GJR_VaR_5_t1', 'Signal1', 'Signal4', 'KOSPI 200_OG', 'KODEX 200_Premium', 'KOSPI 200_DMI14', 'KOSPI 200 lagged_2_return(%)']

selected_features:
['NASDAQ_return(%)', 'KOSPI 200_PPO', 'KOSPI 200 Volume', 'Brent Crude Oil_return(%)', 'GJR_VaR_5_t1', 'Signal1_Buy', 'Signal1_Sell', 'Signal4_Buy', 'Signal4_Sell', 'KOSPI 200_OG', 'KODEX 200_Premium', 'KOSPI 200_DMI14', 'KOSPI 200 lagged_2_return(%)']


### Backward Elimination 실행

In [17]:
# =========================
# 2) RF Backward Elimination
# =========================
# backward_drop_tolerance=1:
# 거의 끝까지 제거 경로를 탐색하기 위한 설정.
# 단, Cell 5의 backward 함수가 "경로 중 최대 G-Mean 변수셋"을 반환해야 의미가 있음.

backward_result = backward_elimination_rf(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid_fs,
    y_valid=y_valid_fs,
    feature_units=feature_units,
    rf_param_list=rf_param_list,
    threshold_grid=threshold_grid,
    backward_drop_tolerance=1,
    min_units_to_keep=1,
    random_state=1,
    n_jobs=-1,
    verbose=True
)

selection_results["backward"] = backward_result

print("\n" + "=" * 80)
print("Backward Elimination 결과")
print("=" * 80)
print("선택 단위 수:", len(backward_result["selected_units"]))
print("실제 컬럼 수:", len(backward_result["selected_features"]))
print("best threshold:", backward_result["best_params"]["threshold"])
print("valid1 gmean:", backward_result["best_metrics"]["gmean"])
print("valid1 recall:", backward_result["best_metrics"]["recall"])
print("valid1 f1:", backward_result["best_metrics"]["f1"])

print("\nselected_units:")
print(backward_result["selected_units"])

print("\nselected_features:")
print(backward_result["selected_features"])


[BACKWARD INITIAL]
초기 선택 단위 수: 24
initial gmean=0.6838 | recall=0.6413 | f1=0.3113 | threshold=0.2

[BACKWARD STEP 1]
현재 선택 단위 수: 24


BACKWARD 1 remove: 100%|██████████| 24/24 [00:16<00:00,  1.47it/s]


제거 단위: Brent Crude Oil_return(%) | gmean=0.6912 | improvement=0.007304 | threshold=0.2  <-- 현재까지 최고

[BACKWARD STEP 2]
현재 선택 단위 수: 23


BACKWARD 2 remove: 100%|██████████| 23/23 [00:15<00:00,  1.53it/s]


제거 단위: return(%) | gmean=0.6894 | improvement=-0.001758 | threshold=0.2

[BACKWARD STEP 3]
현재 선택 단위 수: 22


...

제거 단위: KOSPI 200_PPO_Hist | gmean=0.6661 | improvement=-0.015697 | threshold=0.25

[BACKWARD STEP 22]
현재 선택 단위 수: 3


BACKWARD 22 remove: 100%|██████████| 3/3 [00:02<00:00,  1.27it/s]


제거 단위: KOSPI 200_BB_width | gmean=0.6385 | improvement=-0.027577 | threshold=0.25

[BACKWARD STEP 23]
현재 선택 단위 수: 2


BACKWARD 23 remove: 100%|██████████| 2/2 [00:01<00:00,  1.31it/s]

제거 단위: KOSPI 200 lagged_2_return(%) | gmean=0.6239 | improvement=-0.014592 | threshold=0.3

[BACKWARD 최종 선택]
전체 제거 경로 중 valid1 G-Mean이 최대인 지점 선택
best_seen_units: 11
best_seen_features: 12
best_seen_gmean: 0.720695886757871
best_seen_threshold: 0.2

Backward Elimination 결과
선택 단위 수: 11
실제 컬럼 수: 12
best threshold: 0.2
valid1 gmean: 0.720695886757871
valid1 recall: 0.7065217391304348
valid1 f1: 0.34210526315789475

selected_units:
['Signal4', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'GJR_VaR_5_t1']

selected_features:
['Signal4_Buy', 'Signal4_Sell', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'GJR_VaR_5_t1']


### Stepwise Selection 실행

In [18]:
# =========================
# 3) RF Stepwise Selection
# =========================

stepwise_result = stepwise_selection_rf(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid_fs,
    y_valid=y_valid_fs,
    feature_units=feature_units,
    rf_param_list=rf_param_list,
    threshold_grid=threshold_grid,
    min_delta=0,
    backward_drop_tolerance=0,
    max_units_to_select=None,
    random_state=1,
    n_jobs=-1,
    verbose=True
)

selection_results["stepwise"] = stepwise_result

print("\n" + "=" * 80)
print("Stepwise Selection 결과")
print("=" * 80)
print("선택 단위 수:", len(stepwise_result["selected_units"]))
print("실제 컬럼 수:", len(stepwise_result["selected_features"]))
print("best threshold:", stepwise_result["best_params"]["threshold"])
print("valid1 gmean:", stepwise_result["best_metrics"]["gmean"])
print("valid1 recall:", stepwise_result["best_metrics"]["recall"])
print("valid1 f1:", stepwise_result["best_metrics"]["f1"])

print("\nselected_units:")
print(stepwise_result["selected_units"])

print("\nselected_features:")
print(stepwise_result["selected_features"])


[STEPWISE STEP 1 - FORWARD]
현재 선택 단위 수: 0 | 추가 후보 단위 수: 24


STEPWISE 1 add: 100%|██████████| 24/24 [00:16<00:00,  1.44it/s]


추가 단위: NASDAQ_return(%) | gmean=0.6239 | improvement=nan | recall=0.5761 | f1=0.2536 | threshold=0.3  <-- 현재까지 최고

[STEPWISE STEP 2 - FORWARD]
현재 선택 단위 수: 1 | 추가 후보 단위 수: 23


STEPWISE 2 add: 100%|██████████| 23/23 [00:15<00:00,  1.45it/s]


추가 단위: KOSPI 200_PPO | gmean=0.6646 | improvement=0.040704 | recall=0.6304 | f1=0.2886 | threshold=0.25  <-- 현재까지 최고

[STEPWISE STEP 2 - BACKWARD]
현재 선택 단위 수: 2


...

추가 단위: KOSPI 200_ADX14 | gmean=0.7047 | improvement=-0.018123 | recall=0.6957 | f1=0.3224 | threshold=0.2

[STEPWISE STEP 120 - BACKWARD]
현재 선택 단위 수: 16


STEPWISE 120 remove: 100%|██████████| 16/16 [00:08<00:00,  1.81it/s]


제거 단위: Signal3 | gmean=0.7222 | improvement=0.017525 | threshold=0.2

[STEPWISE STEP 120 - BACKWARD]
현재 선택 단위 수: 15


STEPWISE 120 remove: 100%|██████████| 15/15 [00:08<00:00,  1.68it/s]

제거해도 현재 stepwise 성능이 개선되지 않으므로 backward prune 종료

[STEPWISE] max_steps=120 도달, 종료

[STEPWISE 최종 선택]
전체 stepwise 경로 중 valid1 G-Mean이 최대인 지점 선택
best_seen_units: 9
best_seen_features: 9
best_seen_gmean: 0.7259162873662188
best_seen_threshold: 0.2

Stepwise Selection 결과
선택 단위 수: 9
실제 컬럼 수: 9
best threshold: 0.2
valid1 gmean: 0.7259162873662188
valid1 recall: 0.7065217391304348
valid1 f1: 0.3504043126684636

selected_units:
['NASDAQ_return(%)', 'KOSPI 200 Volume', 'Brent Crude Oil_return(%)', 'KOSPI 200_OG', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200_DMI14', 'USD/KRW_change(%)', 'GJR_VaR_5_t1', 'KODEX 200_Premium']

selected_features:
['NASDAQ_return(%)', 'KOSPI 200 Volume', 'Brent Crude Oil_return(%)', 'KOSPI 200_OG', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200_DMI14', 'USD/KRW_change(%)', 'GJR_VaR_5_t1', 'KODEX 200_Premium']


### valid1 요약 + valid2 비교

In [19]:
# =========================
# 4) valid1 요약 + valid2에서 Forward / Backward / Stepwise 비교
# =========================

required_methods = ["forward", "backward", "stepwise"]
missing_methods = [m for m in required_methods if m not in selection_results]

if len(missing_methods) > 0:
    raise ValueError(f"아직 실행되지 않은 selection result가 있습니다: {missing_methods}")

# =========================
# valid1 내부 결과 요약
# =========================
summary_rows = []

for method_name, result in selection_results.items():
    m = result["best_metrics"]
    p = result["best_params"]

    summary_rows.append({
        "method": method_name,
        "n_units_valid1": len(result["selected_units"]),
        "n_features_valid1": len(result["selected_features"]),
        "threshold_valid1": p["threshold"],
        "gmean_valid1": m["gmean"],
        "recall_valid1": m["recall"],
        "f1_valid1": m["f1"],
        "acc_valid1": m["acc"],
        "selected_units": " | ".join(result["selected_units"]),
        "selected_features": " | ".join(result["selected_features"])
    })

valid1_summary_df = pd.DataFrame(summary_rows).sort_values(
    ["gmean_valid1", "recall_valid1", "f1_valid1", "acc_valid1"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("valid1 결과 요약: Forward / Backward / Stepwise 각각의 내부 선택 결과")
print("valid1 성능은 변수셋 생성용 성능임. 최종 성능으로 해석 금지.")
print("=" * 80)
display(valid1_summary_df)

# =========================
# valid2에서 세 방법 비교
# =========================
X_train_eval = pd.concat([X_train, X_valid_fs], axis=0).reset_index(drop=True)
y_train_eval = pd.concat([y_train, y_valid_fs], axis=0).reset_index(drop=True)

method_comparison_df = evaluate_rf_wrapper_methods_on_valid2(
    selection_results=selection_results,
    X_train_eval=X_train_eval,
    y_train_eval=y_train_eval,
    X_valid2=X_valid_tune,
    y_valid2=y_valid_tune,
    rf_param_list=rf_param_list,
    threshold_grid=threshold_grid,
    random_state=1,
    n_jobs=-1
)

print("\n" + "=" * 80)
print("valid2 방법 비교: RF Forward vs Backward vs Stepwise")
print("valid2 성능은 RF wrapper 방식 선택용 성능임. 최종 test 성능 아님.")
print("=" * 80)

compare_cols = [
    "method", "n_units", "n_features",
    "gmean", "threshold", "acc", "f1", "precision", "recall", "spec",
    "balanced_acc", "mcc", "roc_auc", "pr_auc", "tn", "fp", "fn", "tp",
    "selected_units", "selected_features"
]

display(method_comparison_df[compare_cols])

best_method = method_comparison_df.iloc[0]["method"]
rf_wrapper_final_units = selection_results[best_method]["selected_units"]
rf_wrapper_final_features = selection_results[best_method]["selected_features"]

print("\n" + "=" * 80)
print("RF wrapper 최종 채택 방식")
print("=" * 80)
print("best_method:", best_method)
print("선택 단위 수:", len(rf_wrapper_final_units))
print("실제 컬럼 수:", len(rf_wrapper_final_features))

print("\nselected_units:")
print(rf_wrapper_final_units)

print("\nselected_features:")
print(rf_wrapper_final_features)


valid1 결과 요약: Forward / Backward / Stepwise 각각의 내부 선택 결과
valid1 성능은 변수셋 생성용 성능임. 최종 성능으로 해석 금지.


,method,n_units_valid1,n_features_valid1,threshold_valid1,gmean_valid1,recall_valid1,f1_valid1,acc_valid1,selected_units,selected_features
0,stepwise,9,9,0.2,0.725916,0.706522,0.350404,0.741970,NASDAQ_return(%) | KOSPI 200 Volume | Brent Cr...,NASDAQ_return(%) | KOSPI 200 Volume | Brent Cr...
1,backward,11,12,0.2,0.720696,0.706522,0.342105,0.732334,Signal4 | KOSDAQ_return(%) | NASDAQ_return(%) ...,Signal4_Buy | Signal4_Sell | KOSDAQ_return(%) ...
2,forward,11,13,0.2,0.717527,0.750000,0.324706,0.692719,NASDAQ_return(%) | KOSPI 200_PPO | KOSPI 200 V...,NASDAQ_return(%) | KOSPI 200_PPO | KOSPI 200 V...



valid2 방법 비교: RF Forward vs Backward vs Stepwise
valid2 성능은 RF wrapper 방식 선택용 성능임. 최종 test 성능 아님.


,method,n_units,n_features,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp,selected_units,selected_features
0,backward,11,12,0.669109,0.25,0.677932,0.362205,0.250000,0.657143,0.681293,0.669218,0.243201,0.720323,0.298052,295,138,24,46,Signal4 | KOSDAQ_return(%) | NASDAQ_return(%) ...,Signal4_Buy | Signal4_Sell | KOSDAQ_return(%) ...
1,forward,11,13,0.664855,0.35,0.717694,0.371681,0.269231,0.600000,0.736721,0.668360,0.251962,0.723919,0.296355,319,114,28,42,NASDAQ_return(%) | KOSPI 200_PPO | KOSPI 200 V...,NASDAQ_return(%) | KOSPI 200_PPO | KOSPI 200 V...
2,stepwise,9,9,0.655912,0.35,0.727634,0.368664,0.272109,0.571429,0.752887,0.662158,0.246818,0.715045,0.277377,326,107,30,40,NASDAQ_return(%) | KOSPI 200 Volume | Brent Cr...,NASDAQ_return(%) | KOSPI 200 Volume | Brent Cr...



RF wrapper 최종 채택 방식
best_method: backward
선택 단위 수: 11
실제 컬럼 수: 12

selected_units:
['Signal4', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'GJR_VaR_5_t1']

selected_features:
['Signal4_Buy', 'Signal4_Sell', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'GJR_VaR_5_t1']


In [20]:
'''
# =========================
# 3) 결과 저장
# =========================
output_dir = Path("../../results")
output_dir.mkdir(parents=True, exist_ok=True)

# 전체 변수 기준 0/1 저장: hard voting에 바로 사용하기 좋음
rf_wrapper_selected_df = pd.DataFrame({
    "feature": list(X_train.columns)
})

for method_name, result in selection_results.items():
    selected_set = set(result["selected_features"])
    rf_wrapper_selected_df[f"selected_by_rf_{method_name}"] = [
        1 if f in selected_set else 0
        for f in X_train.columns
    ]

final_set = set(rf_wrapper_final_features)
rf_wrapper_selected_df["selected_by_rf_wrapper"] = [
    1 if f in final_set else 0
    for f in X_train.columns
]
rf_wrapper_selected_df["rf_wrapper_selected_method"] = best_method

# 선택 단위 정보 저장
unit_rows = []
for unit_name, cols in feature_units.items():
    unit_rows.append({
        "unit": unit_name,
        "columns": " | ".join(cols),
        "n_columns": len(cols),
        "selected_by_rf_wrapper": 1 if unit_name in rf_wrapper_final_units else 0,
        "rf_wrapper_selected_method": best_method
    })
rf_wrapper_units_df = pd.DataFrame(unit_rows)

# history 저장
all_step_history = pd.concat(
    [result["step_history"] for result in selection_results.values()],
    axis=0,
    ignore_index=True
)

rf_wrapper_selected_path = output_dir / "rf_wrapper_selected_features.csv"
rf_wrapper_units_path = output_dir / "rf_wrapper_selected_units.csv"
rf_wrapper_method_comparison_path = output_dir / "rf_wrapper_method_comparison_valid2.csv"
rf_wrapper_valid1_summary_path = output_dir / "rf_wrapper_valid1_summary.csv"
rf_wrapper_step_history_path = output_dir / "rf_wrapper_step_history.csv"

rf_wrapper_selected_df.to_csv(rf_wrapper_selected_path, index=False, encoding="utf-8-sig")
rf_wrapper_units_df.to_csv(rf_wrapper_units_path, index=False, encoding="utf-8-sig")
method_comparison_df.to_csv(rf_wrapper_method_comparison_path, index=False, encoding="utf-8-sig")
valid1_summary_df.to_csv(rf_wrapper_valid1_summary_path, index=False, encoding="utf-8-sig")
all_step_history.to_csv(rf_wrapper_step_history_path, index=False, encoding="utf-8-sig")

print("\n저장 완료:")
print(rf_wrapper_selected_path)
print(rf_wrapper_units_path)
print(rf_wrapper_method_comparison_path)
print(rf_wrapper_valid1_summary_path)
print(rf_wrapper_step_history_path)

print("\n주의: test 평가는 아직 하지 않았음. test는 최종 변수셋/모델 확정 후 마지막에만 사용할 것.")
'''

'\n# =========================\n# 3) 결과 저장\n# =========================\noutput_dir = Path("../../results")\noutput_dir.mkdir(parents=True, exist_ok=True)\n\n# 전체 변수 기준 0/1 저장: hard voting에 바로 사용하기 좋음\nrf_wrapper_selected_df = pd.DataFrame({\n    "feature": list(X_train.columns)\n})\n\nfor method_name, result in selection_results.items():\n    selected_set = set(result["selected_features"])\n    rf_wrapper_selected_df[f"selected_by_rf_{method_name}"] = [\n        1 if f in selected_set else 0\n        for f in X_train.columns\n    ]\n\nfinal_set = set(rf_wrapper_final_features)\nrf_wrapper_selected_df["selected_by_rf_wrapper"] = [\n    1 if f in final_set else 0\n    for f in X_train.columns\n]\nrf_wrapper_selected_df["rf_wrapper_selected_method"] = best_method\n\n# 선택 단위 정보 저장\nunit_rows = []\nfor unit_name, cols in feature_units.items():\n    unit_rows.append({\n        "unit": unit_name,\n        "columns": " | ".join(cols),\n        "n_columns": len(cols),\n        "selected_by_rf_